# SparseForrestTomlinLU Bindings Tests

Tests for the `SparseForrestTomlinLU` class exposed via pybind11 bindings.

In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse import csr_matrix, random as sp_random
import time
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '/data/dev/simplinho/build')

try:
    import simplinho as splx
    HAS_SIMPLINHO = True
except ImportError as e:
    HAS_SIMPLINHO = False
    print(f"Warning: simplinho not found. {e}")

try:
    import pulp
    HAS_PULP = True
except ImportError:
    HAS_PULP = False
    print("Warning: pulp not found. Install with: pip install pulp")

try:
    import gurobipy
    HAS_GUROBI = True
except ImportError:
    HAS_GUROBI = False
    print("Gurobi not available (commercial)")

print(f"simplinho: {HAS_SIMPLINHO}, PuLP: {HAS_PULP}, Gurobi: {HAS_GUROBI}")

Gurobi not available (commercial)
simplinho: True, PuLP: True, Gurobi: False


## Test 1: Basic LU Factorization

In [2]:
# Create a simple sparse matrix
import scipy.sparse as sps
n = 5
A_dense = np.array([
    [2.0, 1.0, 0.0, 0.0, 0.0],
    [1.0, 3.0, 1.0, 0.0, 0.0],
    [0.0, 1.0, 4.0, 1.0, 0.0],
    [0.0, 0.0, 1.0, 5.0, 1.0],
    [0.0, 0.0, 0.0, 1.0, 6.0]
])
A_sparse = sps.csc_matrix(A_dense)

print(f"Matrix shape: {A_sparse.shape}")
print(f"Non-zeros: {A_sparse.nnz}")

Matrix shape: (5, 5)
Non-zeros: 13


## Test 2: Create and Factor SparseForrestTomlinLU

In [3]:
lu = splx.SparseForrestTomlinLU()

# Factorize
lu.factor(A_sparse, pivot_rel=1e-12, abs_floor=1e-16, refactor_rook_iters=2)

print(f"supports_inplace_updates: {lu.supports_inplace_updates()}")
print(f"has_updates: {lu.has_updates()}")
print(f"is_rank_deficient: {lu.is_rank_deficient()}")

supports_inplace_updates: True
has_updates: False
is_rank_deficient: False


## Test 3: Solve B*x = b

In [4]:
b = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
x = lu.solve(b)

print(f"Solve result x: {x}")

# Verify: A @ x should equal b
b_check = A_sparse @ x
print(f"A @ x: {b_check}")
print(f"Residual: {np.linalg.norm(b - b_check)}")

Solve result x: [0.30284553 0.39430894 0.51422764 0.54878049 0.74186992]
A @ x: [1. 2. 3. 4. 5.]
Residual: 2.220446049250313e-16


## Test 4: Solve B^T*y = c

In [5]:
c = np.array([0.5, 1.0, 1.5, 2.0, 2.5])
y = lu.solveT(c)

print(f"SolveT result y: {y}")

# Verify: A^T @ y should equal c
c_check = A_sparse.T @ y
print(f"A^T @ y: {c_check}")
print(f"Residual: {np.linalg.norm(c - c_check)}")

SolveT result y: [0.15142276 0.19715447 0.25711382 0.27439024 0.37093496]
A^T @ y: [0.5 1.  1.5 2.  2.5]
Residual: 1.1102230246251565e-16


## Test 5: Sparse RHS Solve

In [6]:
# Sparse RHS: only indices 0 and 3 are non-zero
seed_idx = [0, 3]
seed_val = [1.0, 4.0]

x_sparse = lu.solve_sparse(seed_idx, seed_val)
print(f"Sparse solve result x: {x_sparse}")

print(f"last_solve_pattern_valid: {lu.last_solve_pattern_valid()}")

Sparse solve result x: [ 0.56300813 -0.12601626 -0.18495935  0.86585366 -0.14430894]
last_solve_pattern_valid: False


## Test 6: Forrest-Tomlin Updates

In [7]:
# Create a simple update
# pivot row j=0
j = 0

# u, z, w vectors for the update
u = np.array([0.0, 0.1, 0.0, 0.0, 0.0])
z = np.array([1.0, 0.2, 0.0, 0.0, 0.0])
w = np.array([0.0, 0.3, 0.0, 0.0, 0.0])
alpha = 2.0

success = lu.append_forrest_tomlin_update(j, u, z, w, alpha)
print(f"Update success: {success}")
print(f"has_updates after append: {lu.has_updates()}")

# Get update stats
stats = lu.update_stats()
print(f"Update stats: count={stats.count}, max_z_inf={stats.max_z_inf}, norm_growth_estimate={stats.norm_growth_estimate}")

Update success: True
has_updates after append: True
Update stats: count=1, max_z_inf=1.0, norm_growth_estimate=1.0


## Test 7: Solve with Updates

In [8]:
# Solve after updates
b_with_updates = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
x_with_updates = lu.solve(b_with_updates)
print(f"Solve with updates result x: {x_with_updates}")

Solve with updates result x: [0.28862022 0.37826238 0.51739164 0.54812587 0.74197902]


## Test 8: Config Options

In [9]:
config = splx.SparseLUConfig()
config.use_amd_ordering = True
config.diagonal_equilibration = True
config.iterative_refinement = True
config.iterative_refinement_steps = 3
config.enable_hyper_sparse_rhs = True
config.use_product_form_updates = True
config.enable_solve_oracle = False
config.validate_solves = False

print(f"Config: use_amd_ordering={config.use_amd_ordering}, diagonal_equilibration={config.diagonal_equilibration}, enable_solve_oracle={config.enable_solve_oracle}, validate_solves={config.validate_solves}")

# Create new LU with config
lu_config = splx.SparseForrestTomlinLU()
lu_config.factor_with_config(A_sparse, pivot_rel=1e-12, abs_floor=1e-16, refactor_rook_iters=2, config=config)
print(f"LU with config supports_inplace_updates: {lu_config.supports_inplace_updates()}")

Config: use_amd_ordering=True, diagonal_equilibration=True, enable_solve_oracle=False, validate_solves=False
LU with config supports_inplace_updates: True


## Test 9: Update Failure Reasons

In [10]:
# Try invalid update (bad dimensions)
j_bad = 10  # out of bounds
u_bad = np.array([1.0, 2.0, 3.0])  # wrong size
z_bad = np.array([1.0, 2.0, 3.0])
w_bad = np.array([1.0, 2.0, 3.0])

success_bad = lu.append_forrest_tomlin_update(j_bad, u_bad, z_bad, w_bad, 2.0)
print(f"Invalid update success: {success_bad}")
print(f"Last failure reason: {lu.last_update_failure_reason_message()}")

Invalid update success: False
Last failure reason: Sparse FT update rejected due to dimension or index mismatch


## Test 10: Rank Deficient Matrix

In [11]:
# Create a rank-deficient matrix
A_rank_def = np.array([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0],  # row 2 = 2 * row 1
    [3.0, 6.0, 9.0]   # row 3 = 3 * row 1
])
A_rank_def_sparse = sps.csc_matrix(A_rank_def)

lu_rank_def = splx.SparseForrestTomlinLU()
lu_rank_def.factor(A_rank_def_sparse, pivot_rel=1e-12, abs_floor=1e-16, refactor_rook_iters=2)

print(f"Rank deficient supports_inplace_updates: {lu_rank_def.supports_inplace_updates()}")
print(f"Rank deficient is_rank_deficient: {lu_rank_def.is_rank_deficient()}")
print(f"Rank deficiency: {lu_rank_def.rank_deficiency()}")

Rank deficient supports_inplace_updates: True
Rank deficient is_rank_deficient: True
Rank deficiency: 1
